In [1]:
import trenchripper as tr

import tifffile
import dask
import h5py
import os

import skimage as sk
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask.delayed as delayed
from dask.distributed import fire_and_forget

from parse import compile
from nd2reader import ND2Reader
from distributed.client import futures_of
from time import sleep

import omnipose
from cellpose_omni import models, core
from omnipose.utils import normalize99

from matplotlib import pyplot as plt

/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/omnipose/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# addition of active memory manager
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/de64/scratch/de64/dask"

### Functions For Notebook

In [3]:
def normalize99_uint8(img, lower_pct=0.01, upper_pct=99.99):
    """
    Clip img at the given percentiles, linearly rescale to [0,255],
    and return as uint8.
    """
    # Compute percentile bounds
    p_low, p_high = np.percentile(img, (lower_pct, upper_pct))
    # Rescale intensities into [0, 255]
    img_rescaled = sk.exposure.rescale_intensity(
        img,
        in_range=(p_low, p_high),
        out_range=(0, 255)
    )
    return img_rescaled.astype(np.uint8)

def width_length_from_permeter_area(perimeter,area):
    width = (perimeter-np.sqrt(perimeter**2 - 4*np.pi*area))/np.pi
    length = ((perimeter - (np.pi*width))/2) + width
    return width,length

def nd2_to_tiff(nd2file,fov_metadata_chunk,channels,tiff_metadata):
    #### start reading nd2 files
    with ND2Reader(nd2file) as nd2_input:
        for i,channel in enumerate(channels):
            for fov in fov_metadata_chunk.index.get_level_values(0).unique():
                for j in fov_metadata_chunk.index.get_level_values(1).unique():
                    fov_row = fov_metadata_chunk.loc[fov,j]
                    output_filename = fov_row[channel + " File Names"]
                    output_filename_path = tiff_output_path + "/" + output_filename
                    fovnum = fov_row["old fov"]
                    nd2_image = nd2_input.get_frame_2D(c=i, t=j, v=fovnum)
                    nd2_image = np.array(nd2_image)
                    tifffile.imwrite(output_filename_path,data=nd2_image,dtype="uint16",\
                                    extratags=tiff_metadata)
    return 1

## using fluorescence model AND local otsu for consistency with main pipeline
def segment_omnipose(headpath,img_idx,seg_channel,min_size=50,border_buffer=4,otsu_filter=True,otsu_window_size=15):
    imgpath = headpath + "/hdf5/hdf5_" + str(img_idx) + ".hdf5"
    segpath = headpath + "/fluorsegmentation/segmentation_" + str(img_idx) + ".hdf5"
    with h5py.File(imgpath,"r") as infile:
        img_data = infile[seg_channel][0]

    try:
        norm_img_data = normalize99(img_data)
        use_GPU = core.use_gpu()
        model_name = 'bact_fluor_omni'
        model = models.CellposeModel(gpu=use_GPU, model_type=model_name)
        omni_masks, _, _ = model.eval(norm_img_data,channels=[0,0],rescale=None,mask_threshold=-1,
                transparency=True,flow_threshold=0,
                niter=None,omni=True,cluster=True,resample=True,
                verbose=False,affinity_seg=0,tile=True,
                augment=False)

        unlabeled_composite_mask = omni_masks>0
        unlabeled_composite_mask = ~sk.morphology.binary_closing(~unlabeled_composite_mask)
        omni_masks[~unlabeled_composite_mask] = 0
        omni_masks = sk.morphology.remove_small_objects(omni_masks,min_size=min_size)
        omni_masks = sk.segmentation.clear_border(omni_masks,buffer_size=border_buffer)
        omni_masks,_,_ = sk.segmentation.relabel_sequential(omni_masks)
        
        omni_masks = omni_masks[np.newaxis,np.newaxis,:,:] #compatability with the mother machine code (k,t,y,x)
    except:
        omni_masks = np.zeros((1,1,img_data.shape[1],img_data.shape[2]),dtype="uint8")

    if otsu_filter:
        norm_uint8_img_data = normalize99_uint8(img_data)
        otsu_selem = sk.morphology.disk(otsu_window_size)
        # plt.imshow(sk.filters.rank.otsu(img_data,otsu_selem))
        otsu_mask = norm_uint8_img_data>(sk.filters.rank.otsu(norm_uint8_img_data,otsu_selem))
        otsu_mask = otsu_mask[np.newaxis,np.newaxis,:,:]
        omni_masks = omni_masks*otsu_mask
        # suppose `seg` is your segmentation array, with labels like [0,1,3,7,…]
        omni_masks, _, _ = sk.segmentation.relabel_sequential(omni_masks,offset=1)
        
    with h5py.File(segpath,"w") as outfile:
        outfile["data"] = omni_masks

    return 1

# def segment_nucleoids(file_idx,headpath,cellsegfolder,nucleoidsegfolder,seg_channel,cell_otsu_scaling,\
#                       min_size=400,border_buffer=4):
#     with h5py.File(headpath + '/hdf5/hdf5_' + str(file_idx) + '.hdf5',"r") as infile:
#         img_data = infile[seg_channel][:]
#     with h5py.File(headpath + "/" + cellsegfolder + "/segmentation_" + str(file_idx) + ".hdf5","r") as infile:
#         seg_data = infile["data"][:]

#     img_arr = img_data[0]
#     seg_arr = seg_data[0,0]

#     composite_mask_list = []
#     nucleoid_idx = 0
#     if np.max(seg_arr)>0:
#         for cell_i in range(1,np.max(seg_arr)+1):
#             cell_seg_mask = (seg_arr==cell_i)
#             cell_otsu_thr = sk.filters.threshold_otsu(img_arr[cell_seg_mask])*cell_otsu_scaling
#             composite_mask = cell_seg_mask&(img_arr>cell_otsu_thr)

#             composite_mask = sk.morphology.label(composite_mask).astype("uint8")
#             nucleoid_idx_increment = np.max(composite_mask)
#             composite_mask[composite_mask>0] = composite_mask[composite_mask>0]+nucleoid_idx
#             nucleoid_idx += nucleoid_idx_increment
#             composite_mask_list.append(composite_mask)
#         composite_mask = np.sum(np.stack(composite_mask_list,dtype="uint8"),axis=0,dtype="uint8")
#         ##morphological operations
#         unlabeled_composite_mask = composite_mask>0
#         unlabeled_composite_mask = ~sk.morphology.binary_closing(~unlabeled_composite_mask)
#         composite_mask[~unlabeled_composite_mask] = 0
#         composite_mask = sk.morphology.remove_small_objects(composite_mask,min_size=min_size)
#         composite_mask = sk.segmentation.clear_border(composite_mask,buffer_size=border_buffer)
#         composite_mask,_,_ = sk.segmentation.relabel_sequential(composite_mask)

#         composite_mask = composite_mask[np.newaxis,np.newaxis,:,:]
        
#     else:
#         composite_mask = np.zeros((1,1,img_data.shape[1],img_data.shape[2]),dtype="uint8")

#     with h5py.File(headpath + "/" + nucleoidsegfolder + "/segmentation_" + str(file_idx) + ".hdf5","w") as outfile:
#         outfile["data"] = composite_mask

#     return 1

class regionprops_extractor_agarpad:
    # hacky analyzer for analyzing agarpad data
    def __init__(self,headpath,segmentationdir,analysisdir,intensity_channel_list=None,props_list=['centroid','area'],custom_props_list=[],\
                 intensity_props_list=['mean_intensity'],custom_intensity_props_list=[],props_to_unpack={'centroid':["centroid_y","centroid_x"]},\
                 pixel_scaling_factors={'area':2,'centroid_y': 1,'centroid_x': 1}):
        self.headpath = headpath
        self.intensity_channel_list = intensity_channel_list
        self.intensity_channel_dict = {channel:i for i,channel in enumerate(intensity_channel_list)}
        self.hdf5path = headpath + "/hdf5"
        self.segmentationpath = headpath + "/" + segmentationdir

        self.global_metapath = self.headpath + "/metadata.hdf5"
        self.meta_handle = tr.pandas_hdf5_handler(self.global_metapath)
        fovdf = self.meta_handle.read_df("global",read_metadata=True)
        self.metadata = fovdf.metadata

        self.analysispath = headpath + "/" + analysisdir
        self.props_list = props_list
        self.custom_props_list = custom_props_list
        self.custom_props_str_list = [item.__name__ for item in self.custom_props_list]
        self.intensity_props_list = intensity_props_list
        self.custom_intensity_props_list = custom_intensity_props_list
        self.custom_intensity_props_str_list = [item.__name__ for item in self.custom_intensity_props_list]
        self.props_to_unpack = props_to_unpack
        self.pixel_scaling_factors = pixel_scaling_factors

    def get_file_regionprops(self,file_idx):
        pixel_microns = self.metadata['pixel_microns']

        segmentation_file = self.segmentationpath + "/segmentation_" + str(file_idx) + ".hdf5"
        hdf5_file = self.hdf5path + "/hdf5_" + str(file_idx) + ".hdf5"

        with h5py.File(segmentation_file,"r") as segfile:
            seg_arr = segfile["data"][:]
        if self.intensity_channel_list is not None:
            img_arr_list = []
            with h5py.File(hdf5_file,"r") as hdf5file:
                for intensity_channel in self.intensity_channel_list:
                    img_arr_list.append(hdf5file[intensity_channel][:])
        props_output = []
        for t in range(seg_arr.shape[1]):
            labels = sk.measure.label(seg_arr[0,t])
            ## Measure regionprops of background pixels; will always be marked as the first object
            labels += 1

            #non intensity info first
            non_intensity_rps = sk.measure.regionprops(labels,extra_properties=self.custom_props_list)

            if self.intensity_channel_list is not None:
                intensity_rps_list = []
                for i,intensity_channel in enumerate(self.intensity_channel_list):
                    intensity_rps = sk.measure.regionprops(labels, img_arr_list[i][t],extra_properties=self.custom_intensity_props_list)
                    intensity_rps_list.append(intensity_rps)

            for idx in range(len(non_intensity_rps)):
                rp = non_intensity_rps[idx]
                props_entry = [file_idx, t, idx]
                for prop_key in (self.props_list+self.custom_props_str_list):
                    prop = getattr(rp, prop_key)
                    if prop_key in self.props_to_unpack.keys():
                        prop_out = dict(zip(self.props_to_unpack[prop_key],list(prop)))
                    else:
                        prop_out = {prop_key:prop}
                    for key,value in prop_out.items():
                        if key in self.pixel_scaling_factors.keys():
                            output = value*(pixel_microns**self.pixel_scaling_factors[key])
                        else:
                            output = value
                        props_entry.append(output)

                if self.intensity_channel_list is not None:
                    for i,intensity_channel in enumerate(self.intensity_channel_list):
                        intensity_rps=intensity_rps_list[i]
                        inten_rp = intensity_rps[idx]
                        for prop_key in (self.intensity_props_list+self.custom_intensity_props_str_list):
                            prop = getattr(inten_rp, prop_key)
                            if prop_key in self.props_to_unpack.keys():
                                prop_out = dict(zip(self.props_to_unpack[prop_key],list(prop)))
                            else:
                                prop_out = {prop_key:prop}
                            for key,value in prop_out.items():
                                if key in self.pixel_scaling_factors.keys():
                                    output = value*(pixel_microns**self.pixel_scaling_factors[key])
                                else:
                                    output = value
                                props_entry.append(output)

                props_output.append(props_entry)

        base_list = ['File Index','timepoints','Objectid']

        unpacked_props_list = []
        for prop_key in (self.props_list+self.custom_props_str_list):
            if prop_key in self.props_to_unpack.keys():
                unpacked_names = self.props_to_unpack[prop_key]
                unpacked_props_list += unpacked_names
            else:
                unpacked_props_list.append(prop_key)

        if self.intensity_channel_list is not None:
            for channel in self.intensity_channel_list:
                for prop_key in (self.intensity_props_list+self.custom_intensity_props_str_list):
                    if prop_key in self.props_to_unpack.keys():
                        unpacked_names = self.props_to_unpack[prop_key]
                        unpacked_props_list += [channel + " " + item for item in unpacked_names]
                    else:
                        unpacked_props_list.append(channel + " " + prop_key)

        column_list = base_list + unpacked_props_list

        df_out = pd.DataFrame(props_output, columns=column_list).reset_index()
        file_idx = df_out.apply(lambda x: int(f'{x["File Index"]:08n}{x["timepoints"]:04n}{x["Objectid"]:02n}'), axis=1)

        df_out["File Parquet Index"] = [item for item in file_idx]
        df_out = df_out.set_index("File Parquet Index").sort_index()
        del df_out["index"]

        return df_out

    def analyze_all_files(self,dask_cont):
        fovdf = self.meta_handle.read_df("global",read_metadata=True)
        file_list = fovdf["File Index"].unique().tolist()

        delayed_list = []
        for file_idx in file_list:
            df_delayed = delayed(self.get_file_regionprops)(file_idx)
            delayed_list.append(df_delayed.persist())

        ## filtering out non-failed dataframes ##
        all_delayed_futures = []
        for item in delayed_list:
            all_delayed_futures += futures_of(item)
        while any(future.status == "pending" for future in all_delayed_futures):
            sleep(0.1)

        good_delayed = []
        for item in delayed_list:
            if all([future.status == "finished" for future in futures_of(item)]):
                good_delayed.append(item)

        ## compiling output dataframe ##
        df_out = dd.from_delayed(good_delayed).persist()
        df_out["File Parquet Index"] = df_out.index
        df_out = df_out.set_index("File Parquet Index", drop=True, sorted=False)
        df_out = df_out.repartition(partition_size="25MB").persist()

        fovdf = fovdf.reset_index()
        fovdf["File Merge Index"] = fovdf.apply(lambda x: int(f'{x["File Index"]:08n}{x["timepoints"]:04n}'), axis=1)
        fovdf = fovdf.set_index("File Merge Index")
        fovdf = fovdf.drop(["File Index","timepoints"], axis=1)

        df_out["File Merge Index"] = df_out.apply(lambda x: int(f'{x["File Index"]:08n}{x["timepoints"]:04n}'), axis=1)
        df_out = df_out.reset_index(drop=True)
        df_out = df_out.set_index("File Merge Index", sorted=True)

        df_out = df_out.join(fovdf)
        df_out = df_out.set_index("File Parquet Index",sorted=True)

        dd.to_parquet(
            df_out,
            self.analysispath,
            engine="pyarrow",
            compression="gzip",
            write_metadata_file=True,
        )

# def nucleoid_cellid(nuc_df_block,pixel_microns):
#     nuc_df_block = nuc_df_block.sort_index()
#     idx = nuc_df_block.iloc[0]["File Index"]
#     with h5py.File(headpath + "/nucleoidsegmentation/segmentation_" + str(idx) + ".hdf5","r") as infile:
#         data = (infile['data'][0,0])
#     nucleoid_coords = nuc_df_block[["centroid_y","centroid_x"]].apply(lambda x: (int(np.round(x["centroid_y"]/pixel_microns)),int(np.round(x["centroid_x"]/pixel_microns))), axis=1)
#     assgined_cell = nucleoid_coords.apply(lambda x: data[x])
#     nuc_df_block["Cell Objectid"] = assgined_cell
#     return nuc_df_block

In [12]:
dask_controller = tr.trcluster.dask_controller(
    walltime="1:00:00",
    local=False,
    n_workers=30,
    n_workers_min=30,
    memory="4GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

50m
1:00:00


In [13]:
dask_controller.displaydashboard()

### Make Flat Fields
- Not applying flatfield since no intensity quantification

In [5]:
# tr.generate_flatfield("/home/de64/scratch/de64/sync_folder/2023-09-10_Flat_Fields_Iris_9_100x_MRD31905/BFP-Penta.nd2","/home/de64/scratch/de64/sync_folder/2023-09-10_Flat_Fields_Iris_9_100x_MRD31905/BFP-Penta.tiff")
# tr.generate_flatfield("/home/de64/scratch/de64/sync_folder/2023-09-10_Flat_Fields_Iris_9_100x_MRD31905/GFP-Penta.nd2","/home/de64/scratch/de64/sync_folder/2023-09-10_Flat_Fields_Iris_9_100x_MRD31905/GFP-Penta.tiff")

In [6]:
# dark_outputpath = "/home/de64/scratch/de64/sync_folder/2022-12-26_Flat_Fields_Iris_9_40x/40x_DarkImage.tiff"
# BFP_outputpath = "/home/de64/scratch/de64/sync_folder/2023-09-10_Flat_Fields_Iris_9_100x_MRD31905/BFP-Penta.tiff"
# GFP_outputpath = "/home/de64/scratch/de64/sync_folder/2023-09-10_Flat_Fields_Iris_9_100x_MRD31905/GFP-Penta.tiff"

# fflist = [dark_outputpath,BFP_outputpath,GFP_outputpath]

# fig, axs = plt.subplots(figsize=(15, 20), nrows= 1, ncols = 3)

# for i in range(3):
#     axs[i].imshow(tifffile.imread(fflist[i]))
#     axs[i].set_title(fflist[i].split('/')[-1].split('_')[0])
# plt.tight_layout()

### Convert Nd2 Files to tiff

In [16]:
# convert nd2 files to tiff (custom) of format t{timepoints:d}xy{fov:d}c{channel:d}.tif
# record mapping from condition to fov range in secondary dataframe

nd2_path = "/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads/"
tiff_output_path = "/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads/tiff"

tr.writedir(tiff_output_path, overwrite=True)

nd2_files = []
for root, _, files in os.walk(nd2_path):
    nd2_files.extend([os.path.join(root, f) for f in files if ".nd2" in os.path.splitext(f)[1]])


ndmeta_handle = tr.nd_metadata_handler(nd2_files[0],ignore_fovmetadata=False)
exp_metadata,fov_metadata = ndmeta_handle.get_metadata()
tiff_metadata = [(65326,'d',1,exp_metadata['pixel_microns'],True)]
channels = exp_metadata["channels"]

parsestr = "DE{Strain:d}-{Replicate:d}_{Concentration:d}nM.nd2"
parser = compile(parsestr)

running_fov_idx = 0

all_fov_metadata = []
dask_futures = {}
for nd2_i,nd2_file in enumerate(nd2_files):
    ndmeta_handle = tr.nd_metadata_handler(nd2_file,ignore_fovmetadata=False)
    exp_metadata,fov_metadata = ndmeta_handle.get_metadata()
    new_fov_series = fov_metadata.reset_index()["fov"] + running_fov_idx
    fov_metadata = fov_metadata.reset_index()
    fov_metadata = fov_metadata.rename(columns={"fov": "old fov"})
    fov_metadata["fov"] = new_fov_series
    fov_metadata = fov_metadata.set_index(["fov","timepoints"])
    
    for i,channel in enumerate(exp_metadata['channels']):
        file_name_series = fov_metadata.reset_index().apply(lambda x: "t" + str(int(x["timepoints"])) + "xy" + str(int(x["fov"])) + "c" + str(i) + ".tif", axis=1)
        file_name_series.index = fov_metadata.index
        fov_metadata[channel + " File Names"] = file_name_series

    match = parser.search(nd2_file)
    # Add to dictionary
    parser_dict = match.named
    for key, value in parser_dict.items():
        fov_metadata[key] = value

    all_fov_metadata.append(fov_metadata)
    running_fov_idx += len(fov_metadata)

    dask_controller.futures["nd2 Convert: " + str(nd2_i)] = dask_controller.daskclient.submit(nd2_to_tiff,nd2_file,fov_metadata,channels,tiff_metadata)
all_fov_metadata = pd.concat(all_fov_metadata)
dask_controller.daskclient.gather([future for key,future in dask_controller.futures.items() if "nd2 Convert" in key]);
all_fov_metadata.to_pickle(tiff_output_path + "/conversion_metadata.pkl")

### Tiff Extraction

In [4]:
# headpath = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads'
# tiff_output_path = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads/tiff'

In [18]:
headpath = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads'
tiff_output_path = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads/tiff'
channels = ["mCherry"]

tiff_extractor = tr.tiff_extractor(tiff_output_path,headpath,channels)

In [19]:
tiff_extractor.inter_set_params()

/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/tables/path.py:137: NaturalNameWarning: object name is a Python keyword: 'global'; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/de64/TrenchRipper/trenchripper/utils.py:86: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['channel_paths'], dtype='object')]

  store.put(key, df)
/home/de64/TrenchRipper/trenchripper/utils.py:93: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.metadata = store.get_storer(key).attrs.metadata


interactive(children=(SelectMultiple(description='fov_list', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12…

In [20]:
tiff_extractor.inter_set_flatfieldpaths()

In [21]:
tiff_extractor.extract(dask_controller)

/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/tables/path.py:137: NaturalNameWarning: object name is a Python keyword: 'global'; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/de64/TrenchRipper/trenchripper/utils.py:86: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['channel_paths'], dtype='object')]

  store.put(key, df)
/home/de64/TrenchRipper/trenchripper/utils.py:93: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.metadata = store.get_storer(key).attrs.metadata
/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/tables/path.py:137: NaturalNameWarning: object name is a Python keyword: 'global'; you will not

In [22]:
dask_controller.shutdown()

Done.


### mCherry Segmentation (Omnipose)
- Note I include an additional otsu threshold to produce segmentation consistent with the mother machine pipeline

In [5]:
# dask_controller = tr.trcluster.dask_controller(
#     walltime="2:00:00",
#     local=False,
#     n_workers=10,
#     n_workers_min=10,
#     queue='gpu_quad',
#     memory="8GB",
#     working_directory=dask_wd,
#     job_extra_directives=["--gres", "gpu:teslaV100s:1"]

# )
# dask_controller.startdask()

dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=8,
    n_workers_min=8,
    queue='gpu_paulsson',
    memory="8GB",
    working_directory=dask_wd,
    job_extra_directives=["--gres", "gpu:1"],
    account="paulsson_jmp30_contrib"
)
dask_controller.startdask()

110m
2:00:00


In [6]:
dask_controller.displaydashboard()

In [7]:
headpath = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads'
seg_channel = "mCherry"
overwrite = False #current run was false

tr.writedir(headpath + "/fluorsegmentation", overwrite=overwrite)
meta_handle = tr.pandas_hdf5_handler(headpath + "/metadata.hdf5")
file_idx_list = np.unique(meta_handle.read_df("global")["File Index"])

for file_idx in file_idx_list:
    dask_controller.futures["Omnipose: " + str(file_idx)] = dask_controller.daskclient.submit(segment_omnipose,headpath,file_idx,seg_channel)
    fire_and_forget(dask_controller.futures["Omnipose: " + str(file_idx)])
dask_controller.daskclient.gather([future for key,future in dask_controller.futures.items() if "Omnipose" in key]);

In [8]:
dask_controller.shutdown()

Done.


### Regionprops Parallelizer

In [9]:
dask_controller = tr.trcluster.dask_controller(
    walltime="1:00:00",
    local=False,
    n_workers=15,
    n_workers_min=15,
    queue='short',
    memory="8GB",
    working_directory=dask_wd,
    job_extra_directives=[]

)
dask_controller.startdask()

50m
1:00:00


In [11]:
dask_controller.displaydashboard()

In [12]:
headpath = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads'

cell_analyzer = regionprops_extractor_agarpad(
    headpath,
    "fluorsegmentation",
    "cell_analysis",
    intensity_channel_list=['mCherry'],
    props_list=["centroid","area","major_axis_length","minor_axis_length","perimeter"],
    pixel_scaling_factors={'area': 2, 'centroid_y': 1, 'centroid_x': 1, 'major_axis_length': 1, 'minor_axis_length': 1, 'perimeter': 1},
)

/home/de64/TrenchRipper/trenchripper/utils.py:93: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.metadata = store.get_storer(key).attrs.metadata


In [13]:
cell_analyzer.analyze_all_files(dask_controller)

/home/de64/TrenchRipper/trenchripper/utils.py:93: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.metadata = store.get_storer(key).attrs.metadata
/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/dask_expr/_collection.py:3225: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map or apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, meta=(None, 'int64'))

  warnings.warn(meta_warning(meta))


### Export Final Dataframe

In [16]:
# dask_controller = tr.trcluster.dask_controller(
#     walltime="1:00:00",
#     local=False,
#     n_workers=10,
#     n_workers_min=10,
#     queue='short',
#     memory="8GB",
#     working_directory=dask_wd,
#     job_extra_directives=[]

# )
# dask_controller.startdask()

50m
1:00:00


2025-07-04 15:03:01,233 - distributed.batched - INFO - Batched Comm Closed <TCP (closed)  local=tcp://10.120.17.241:44269 remote=tcp://10.120.17.241:40836>
Traceback (most recent call last):
  File "/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/distributed/batched.py", line 115, in _background_send
    nbytes = yield coro
  File "/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/tornado/gen.py", line 769, in run
    value = future.result()
  File "/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/distributed/comm/tcp.py", line 262, in write
    raise CommClosedError()
distributed.comm.core.CommClosedError


In [18]:
# dask_controller.displaydashboard()

In [29]:
## assign cellids to the nucleoid df

In [23]:
headpath = '/home/de64/scratch/de64/sync_folder/2025-07-03_DE828_AHL_Titration_Agarpads'

cell_df = dd.read_parquet(headpath + "/cell_analysis",calculate_divisions=True).compute()
cell_df = cell_df[cell_df["Objectid"] != 0]

In [24]:
### filter parameters
mCherry_intensity_thr = 750

cell_df_fov_idx = cell_df.apply(lambda x: int(f'{x["fov"]:08n}{x["timepoints"]:04n}{x["Objectid"]:02n}'), axis=1)
cell_df["FOV Parquet Index"] = cell_df_fov_idx
cell_df = cell_df.set_index("FOV Parquet Index")

## extra filters for true cells
cell_df = cell_df[(cell_df["mCherry mean_intensity"]>mCherry_intensity_thr)]

conversion_metadata_df = pd.read_pickle(headpath + '/tiff/conversion_metadata.pkl').reset_index()[["fov","Strain","Replicate","Concentration"]].set_index("fov")
cell_df = cell_df.join(conversion_metadata_df,on="fov")

width, length = width_length_from_permeter_area(cell_df["perimeter"],cell_df["area"])
volume = np.pi * ((1/4)*(width**2)*(length-width) + (1/6)*(width**3))
## add the length, width volume estimates
cell_df["Width"] = width
cell_df["Length"] = length
cell_df["Volume"] = volume

cell_df.to_pickle("/home/de64/group/de64/CRISPRi_Libraries/dev_notebooks/2024-11-23_Figure_Notebooks/Data/FusA_Knockdowns/Imaging/Final_Output_df.pkl")
cell_df.to_csv("/home/de64/group/de64/CRISPRi_Libraries/dev_notebooks/2024-11-23_Figure_Notebooks/Data/FusA_Knockdowns/Imaging/Final_Output_df.csv")

/home/de64/micromamba/envs/agarpad/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [26]:
dask_controller.shutdown()

Done.
